# E2E Part 3 — OSD, adaptive routing, and external SURT

This notebook implements Sections 6–9 of `docx/steps.md`. It requires a persisted Phase 3 development dataset and a working WhisperRT baseline. The direct SURT and routed SURT cells are intentionally gated by an external decoder factory.

In [ ]:
%cd /kaggle/working/English-Streaming-ASR
from pathlib import Path
ROOT = Path.cwd()
assert (ROOT / 'src').is_dir()

# 6. Phase 5 — OSD on the real generated development set

In [ ]:
from pathlib import Path

Path('kaggle_osd.yaml').write_text('''\
data:
  manifest_root: /kaggle/working/English-Streaming-ASR/outputs/synthetic_overlap_dev
  splits: [test]
''')
print(Path('kaggle_osd.yaml').read_text())

!python scripts/evaluate_osd.py \
  --config configs/overlap_detection.yaml \
  --override kaggle_osd.yaml

# 7. Phase 6 — adaptive routing controls

Run all three modes on exactly the same test manifest. `always_normal` and `predicted` are deployable candidates; `oracle` is a non-deployable diagnostic that uses synthetic ground truth.

In [ ]:
from pathlib import Path

Path('kaggle_adaptive.yaml').write_text('''\
data:
  manifest: /kaggle/working/English-Streaming-ASR/outputs/synthetic_overlap_dev/manifests/test.jsonl
  max_samples: null
branches:
  overlap: shared_normal
''')
print(Path('kaggle_adaptive.yaml').read_text())

In [ ]:
!python scripts/run_adaptive.py \
  --config configs/adaptive.yaml \
  --override kaggle_adaptive.yaml \
  --validate-only

!python scripts/run_adaptive.py \
  --config configs/adaptive.yaml \
  --override kaggle_adaptive.yaml \
  --max-samples 1 \
  --modes always_normal oracle predicted

If the one-mixture control run is valid, remove `--max-samples 1` and save every routing trace, metrics file, event file, report, and override. The current `shared_normal` setting is a routing control, not multi-talker recognition.

In [ ]:
# Run only after the one-mixture control run succeeds.
# !python scripts/run_adaptive.py \
#   --config configs/adaptive.yaml \
#   --override kaggle_adaptive.yaml \
#   --modes always_normal oracle predicted

# 8. External SURT integration gate

Before enabling these cells, provide a version-pinned official SURT/Icefall checkout, checkpoint, BPE/tokenizer assets, and a tested project factory. The factory must return `callable(samples: numpy.ndarray, sample_rate: int) -> streams`. This repository does not provide it.

Develop the adapter in a correction branch outside this Kaggle execution notebook. Add an integration test that performs one short incremental decode. Do not replace SURT with a different model.

## 8.3 Phase 7 private override

Replace the bracketed Kaggle input paths only after the factory and checkpoint are verified. Keep this override private/uncommitted if it contains account-specific input paths.

In [ ]:
from pathlib import Path

# Replace both placeholders before executing the validation/run cells below.
Path('kaggle_surt.yaml').write_text('''\
data:
  manifest: /kaggle/working/English-Streaming-ASR/outputs/synthetic_overlap_dev/manifests/test.jsonl
model:
  checkpoint: /kaggle/input/<your-surt-checkpoint-dataset>/<checkpoint-file>
  decoder_factory: streaming_asr.integrations.surt:build_decoder
''')
print(Path('kaggle_surt.yaml').read_text())

## 8.4 Phase 7 direct multi-talker run

Uncomment only after replacing the placeholders and implementing the factory. The adapter must be described as low-latency windowed unless the external decoder has verified native causal state.

In [ ]:
# !python scripts/run_multitalker.py \
#   --config configs/multitalker.yaml \
#   --override kaggle_surt.yaml \
#   --validate-only
#
# !python scripts/run_multitalker.py \
#   --config configs/multitalker.yaml \
#   --override kaggle_surt.yaml \
#   --max-samples 1

# 9. E2E route: WhisperRT normal branch + SURT overlap branch

Enable this only after the direct Phase 7 run is valid. Compare `always_normal`, `oracle`, and `predicted` on the same test set; oracle is never deployable.

In [ ]:
from pathlib import Path

Path('kaggle_adaptive_surt.yaml').write_text('''\
data:
  manifest: /kaggle/working/English-Streaming-ASR/outputs/synthetic_overlap_dev/manifests/test.jsonl
branches:
  overlap: surt2
multitalker:
  decoder_factory: streaming_asr.integrations.surt:build_decoder
  checkpoint: /kaggle/input/<your-surt-checkpoint-dataset>/<checkpoint-file>
''')
print(Path('kaggle_adaptive_surt.yaml').read_text())

In [ ]:
# Remove comments only after Phase 7 has succeeded with the same factory/checkpoint.
# !python scripts/run_adaptive.py \
#   --config configs/adaptive.yaml \
#   --override kaggle_adaptive_surt.yaml \
#   --max-samples 1 \
#   --modes always_normal oracle predicted

Continue with `notebooks/13_surt_training_evaluation.ipynb` only after saving all direct and routed prediction records.